# 03. Fintech campaign: two factors at once (2x2 factorial)

**Sector:** fintech (CRM). **Decision:** how to design the activation campaign?
Two binary factors are in play at the same time:

- **cashback** (offer vs. not), factor `A`
- **send time** (morning vs. evening), factor `B`

Instead of testing one at a time, a **2x2 factorial** design measures, in a
single experiment, both **main effects** and the **interaction** (does cashback
pay off more when sent in the evening?). Theory:
[II. Designs](../../../docs/en/theory/02-designs.md) (topic 7) and
[III. Estimation](../../../docs/en/theory/03-estimation.md) (topic 12).


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


def _find_data():
    """Locate examples/use_cases/data whether run from the notebook or the root."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "data", base / "examples" / "use_cases" / "data"):
            if (cand / "ecommerce_checkout.csv").exists():
                return cand
    raise FileNotFoundError("Could not locate examples/use_cases/data")


DATA = _find_data()

df = pd.read_csv(DATA / "fintech_crm.csv")
print(df.shape)
df.head()


## 1. Factorial design and the outcome

`FactorialDesign(factors=["cashback", "send_time"], n_per_cell=1000)` draws the
`2^2 = 4` cells with equal size. The outcome (activations) is generated from the
assigned factors, with known coefficients in a `{0,1}` coding:

```
activations = 30 + 4*cashback + 2*send_time + 3*(cashback*send_time) + noise
```

Note that the simulation's `{0,1}` coding is **not** the same scale as the `±1`
contrasts the estimator uses. The theory (topic 12) shows the estimated main
effect is `b + half the interaction`. So we expect to recover:

- cashback: `4 + 3/2 = 5.5`
- send_time: `2 + 3/2 = 3.5`
- interaction: `3/2 = 1.5`


In [ ]:
from skxperiments.core.assignment import FactorialAssignment
from skxperiments.design.factorial import FactorialDesign

design = FactorialDesign(factors=["cashback", "send_time"], n_per_cell=1000, seed=303)
assignment = design.randomize(df[["customer_id", "tenure_months"]].copy())

A = assignment.data_["cashback"].to_numpy()
B = assignment.data_["send_time"].to_numpy()
data = assignment.data_.copy()
data["activations"] = 30 + 4 * A + 2 * B + 3 * (A * B) + df["noise"].to_numpy()

assignment = FactorialAssignment(
    data=data, design=design, factor_cols=assignment.factor_cols,
    cell_sizes=assignment.cell_sizes_, seed=303,
)
assignment.data_.groupby(["cashback", "send_time"])["activations"].mean().round(2)


In [ ]:
from skxperiments.estimators.factorial_estimator import FactorialEstimator

result = FactorialEstimator(outcome_col="activations").fit(assignment).estimate()
expected = {"cashback": 5.5, "send_time": 3.5, "cashback:send_time": 1.5}
rows = []
for key, value in result.effects.items():
    name = ":".join(key)
    rows.append({"effect": name, "estimated": round(value, 3), "expected": expected[name]})
pd.DataFrame(rows)


In [ ]:
from skxperiments.reporting import plot_interaction

ax = plot_interaction(result)
ax.set_title("Main effects and interaction")
ax.figure


## Decision

All three effects are recovered close to expectation (`5.5`, `3.5`, `1.5`). The
**positive interaction** confirms that cashback and evening send reinforce each
other: together they yield more than the sum of the isolated effects would
suggest. The campaign recommendation is to combine both factors at the high
level. A "one factor at a time" design would never reveal this synergy.

Next step: [04. Re-randomization in logistics](04_logistics_rerandomization.ipynb).
